# Lasso Regression — Intuition

**Goal.** Build a mental picture of Lasso *before* touching any math. Three questions, answered with pictures only:

1. What is the L1 penalty, and how does it differ from Ridge's L2 penalty in shape?
2. Why does an L1 penalty make some coefficients **exactly zero**, while L2 only shrinks them?
3. What does the Lasso regularisation path look like, and what is *variable selection*?

No formulas here. The math lives in `02_mathematics.ipynb`.

**One-line preview.** Lasso = OLS with a penalty on the *sum of absolute values* of the coefficients. The geometry of that penalty is a diamond with corners on the axes — so the fitted coefficients tend to land exactly *on* an axis. Coefficients on an axis are coefficients that equal zero. Lasso is therefore the simplest model that does **regression and feature selection in a single optimisation**.

**Prerequisites.** `03_ridge_regression/01_intuition.ipynb` — the regularised-regression framework with one tuning knob $\lambda$. We will compare side-by-side with Ridge throughout.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Two penalties, two shapes

Ridge and Lasso both add a penalty to the OLS loss. The only difference is *which* penalty.

Let the coefficient vector have two entries $\theta$ = ($\theta_1$, $\theta_2$).

- **Ridge penalty.** Sum of squared coefficients: $\theta_1$^2 + $\theta_2$^2. The set of $\theta$ where this equals 1 is a **circle**.
- **Lasso penalty.** Sum of absolute values: |$\theta_1$| + |$\theta_2$|. The set of $\theta$ where this equals 1 is a **diamond** (rotated square with corners on the axes).

Picture them side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True)

# Ridge: circle
t = np.linspace(0, 2 * np.pi, 200)
axes[0].plot(np.cos(t), np.sin(t), color="steelblue", lw=2)
axes[0].set_title("Ridge:  θ₁² + θ₂² = 1  (circle)")

# Lasso: diamond
axes[1].plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], color="crimson", lw=2)
axes[1].set_title("Lasso:  |θ₁| + |θ₂| = 1  (diamond)")

for ax in axes:
    ax.axhline(0, color="black", lw=0.5)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_aspect("equal")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
    ax.set_xlabel("θ₁")
axes[0].set_ylabel("θ₂")
plt.show()

**Reading.** Both shapes are *convex regions* of the same volume, so they regularise *similarly* in size. The crucial visual difference: the diamond has **corners on the axes**, the circle does not. That is the entire reason the Lasso produces zero coefficients.

## 2. Why the diamond shape forces zero coefficients

Recall (`03_ridge_regression/02_mathematics.ipynb` §5) that Ridge can be written as:

> minimise the OLS loss   subject to   $\|\theta\|_2^2$ $\le$ t.

Lasso has the same shape — only the constraint is replaced:

> minimise the OLS loss   subject to   $\|\theta\|_1$ $\le$ t,    where  $\|\theta\|_1$ = |$\theta_1$| + |$\theta_2$| + ... + |$\theta_p$|.

Picture: draw the OLS loss as elliptical contours centred on the unregularised solution $\hat{\theta}_{OLS}$, and draw the constraint region. The Lasso solution is the **first tangency point** as the loss contours grow outward from $\hat{\theta}_{OLS}$.

- With a **circle** (Ridge), tangency happens at a *generic* point — usually no coordinate is exactly zero.
- With a **diamond** (Lasso), the corners stick out toward the axes, so the first tangency is *very likely* to be at a corner — and corners lie on the axes, i.e. *some coordinates of $\theta$ are exactly zero*.

In [ ]:
# OLS unregularised solution and its loss contours.
theta_ols = np.array([1.4, 0.45])
Sigma = np.array([[1.0, 0.7], [0.7, 1.0]])  # shape of the loss ellipsoid

Ti, Tj = np.meshgrid(np.linspace(-1.7, 2.2, 200), np.linspace(-1.7, 1.7, 200))
diff = np.stack([Ti.ravel() - theta_ols[0], Tj.ravel() - theta_ols[1]], axis=1)  # (n_pts, 2)
loss = ((diff @ Sigma) * diff).sum(axis=1).reshape(Ti.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)

# Ridge panel
axes[0].contour(Ti, Tj, loss, levels=10, colors="steelblue", alpha=0.5)
t = np.linspace(0, 2 * np.pi, 200)
r = 0.7
axes[0].plot(r * np.cos(t), r * np.sin(t), color="steelblue", lw=2)
axes[0].plot(*theta_ols, "k+", markersize=12, label="θ̂_OLS")
axes[0].plot(0.55, 0.42, "ro", markersize=10, label="Ridge θ̂")
axes[0].set_title("Ridge:  tangent point inside a quadrant\n→ both coefficients non-zero")

# Lasso panel
axes[1].contour(Ti, Tj, loss, levels=10, colors="crimson", alpha=0.5)
rL = 0.8
axes[1].plot([rL, 0, -rL, 0, rL], [0, rL, 0, -rL, 0], color="crimson", lw=2)
axes[1].plot(*theta_ols, "k+", markersize=12, label="θ̂_OLS")
axes[1].plot(rL, 0, "ro", markersize=10, label="Lasso θ̂")
axes[1].set_title("Lasso:  tangent point at the (θ₁, 0) corner\n→ θ₂ exactly zero")

for ax in axes:
    ax.axhline(0, color="black", lw=0.5)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_aspect("equal")
    ax.set_xlim(-1.7, 2.2); ax.set_ylim(-1.7, 1.7)
    ax.set_xlabel("θ₁"); ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("θ₂")
plt.suptitle("The same OLS loss; only the constraint shape changes")
plt.tight_layout()
plt.show()

**Reading.** Same elliptical loss in both panels. The Ridge ball happens to be tangent somewhere off the axes, so both coefficients are non-zero. The Lasso diamond, in contrast, has a corner pointing toward the (positive-$\theta_1$, zero-$\theta_2$) direction — so the ellipse hits that corner first. The fitted $\hat{\theta}$ has $\hat{\theta}_{2}$ = 0 *exactly*, not just approximately.

In high dimensions, every face of the L1-diamond corresponds to setting some subset of coefficients to zero — and the higher the dimension p, the more faces and corners there are. So Lasso solutions are typically **very sparse** when p is large.

## 3. Lasso on a sparse signal — variable selection in action

Generate data where only a handful of features are truly informative — the rest are noise. Fit OLS, Ridge, and Lasso on the same data and compare their coefficient vectors against the truth.

In [ ]:
n, p = 200, 30
X = rng.normal(size=(n, p))

# Only the first 5 features are informative.
theta_true = np.zeros(p)
theta_true[:5] = [3.0, -2.0, 1.5, 1.0, -0.8]

y = X @ theta_true + rng.normal(0, 0.5, size=n)

# Fit three models.
lasso = Lasso(alpha=0.05).fit(X, y)
ridge = Ridge(alpha=5.0).fit(X, y)
ols   = np.linalg.lstsq(X, y, rcond=None)[0]

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, (name, est, c) in zip(
    axes,
    [("OLS", ols, "black"), ("Ridge", ridge.coef_, "steelblue"), ("Lasso", lasso.coef_, "crimson")],
):
    ax.bar(np.arange(p), est, color=c, alpha=0.85, label=name)
    # Overlay the truth as black dots.
    ax.plot(np.arange(p), theta_true, "ko", markersize=4, label="true θ")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel("coefficient")
    ax.set_title(f"{name}    (non-zero: {int(np.sum(np.abs(est) > 1e-6))} / {p})")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("feature index j")
plt.tight_layout()
plt.show()

**Reading.**

- **OLS** uses *all* 30 features. The first 5 land near the truth, but the other 25 each pick up some small noise-driven coefficient — none is exactly zero. That noise is what makes OLS variance large in high dimensions.
- **Ridge** shrinks every coefficient toward zero — including the true ones, which now sit slightly below their true values. Still, no coefficient is *exactly* zero. Ridge cannot say "this feature is irrelevant".
- **Lasso** does what we want: keeps the 5 true features (slightly under-shrunk, like Ridge) and **kills the other 25 to exactly zero**. The picture matches the truth in shape, not just in magnitude.

## 4. The Lasso path — coefficients enter / leave one at a time

Like Ridge, Lasso has a regularisation path: every coefficient is a function of $\lambda$. Two visual differences from Ridge:

- **Piecewise linear, not smooth.** The Lasso path consists of straight-line segments connected at *kinks* — each kink is the moment one feature joins or leaves the active set.
- **Hard zeros on the right.** For large enough $\lambda$, more and more coefficients are pinned to *exactly* zero; eventually all of them are zero (the model predicts the mean of y).

In [ ]:
alphas = np.logspace(-3, 0.5, 60)
coefs_lasso = np.array([Lasso(alpha=a, max_iter=20000).fit(X, y).coef_ for a in alphas])
coefs_ridge = np.array([Ridge(alpha=a * n).fit(X, y).coef_ for a in alphas])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for j in range(p):
    is_true = j < 5
    color   = "crimson" if is_true else "lightgray"
    lw      = 1.6 if is_true else 0.8
    axes[0].plot(alphas, coefs_ridge[:, j], color=color, lw=lw)
    axes[1].plot(alphas, coefs_lasso[:, j], color=color, lw=lw)

for ax, title in zip(axes, ["Ridge — smooth shrinkage", "Lasso — piecewise linear, zeros emerge"]):
    ax.set_xscale("log")
    ax.set_xlabel("λ (log)")
    ax.set_title(title)
    ax.axhline(0, color="black", lw=0.5)
axes[0].set_ylabel("coefficient value")

# legend
axes[1].plot([], [], color="crimson", label="true non-zero features")
axes[1].plot([], [], color="lightgray", label="noise features")
axes[1].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

**Reading.**

- **Ridge (left)** — all 30 curves are smooth and all stay non-zero everywhere. The grey (noise) curves do shrink fast, but only the limit $\lambda$ → $\infty$ flattens them to zero.
- **Lasso (right)** — every grey curve hits the zero line at *some* finite $\lambda$ and stays there. The red (true) curves stay non-zero for longer. As $\lambda$ grows, the model gracefully drops features one at a time until only the strongest ones survive — and eventually even those get dropped.

The window where the *red* curves are still active and the *grey* curves are all zero is the sweet spot for variable selection.

## Takeaway

- **What changes from Ridge:** the penalty switches from sum-of-squared coefficients (L2) to sum-of-absolute-values (L1).
- **Geometric consequence:** the constraint region is a diamond, not a circle. Diamonds have corners on the axes, so the loss contour very often touches a corner — making some coefficients **exactly zero**.
- **What you gain:** variable selection. Lasso simultaneously fits the model and decides which features to keep.
- **What you trade away:** no closed-form solution. The math is non-smooth at $\theta_j$ = 0, so the optimisation needs more care (`03_optimization.ipynb`). The coefficient path is piecewise linear with kinks instead of smooth curves.
- **What you tune:** $\lambda$, just like Ridge. Cross-validation still chooses it.

Next: `02_mathematics.ipynb` writes down the L1 loss, derives the **subgradient** optimality conditions (since the absolute value is not differentiable at zero), and proves the key identity that drives every Lasso algorithm — the **soft-thresholding** formula.